# Autoresearch feature discovery

**Difficulty:** Advanced

Turn free text into a table of numbers for a supervised ML model — using TypeSafe questions instead of hand-engineered features. A `Score` question becomes two numeric columns (mean level and spread); a `Noul` becomes one (a probability). The "autoresearch" part — an LLM proposing new questions based on model error, over several rounds — is genuinely a separate ML-training pipeline; this cookbook is trimmed to the featurization step alone.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/autoresearch_feature_discovery (click to expand)</summary>

# Autoresearch feature discovery

> Runs an autoresearch loop that proposes TypeSafe questions, converts free text into numeric features, and uses model errors to improve a supervised CatBoost regressor.

CatBoost needs a table of numbers; a tasting note is not one. This cookbook builds the table out of questions about the note, and none of them are written by hand: an LLM proposes the questions, TypeSafe answers them for every row, and CatBoost trains on the answers. The *autoresearch* part is what comes next — CatBoost reports which questions it used and which rows it still gets wrong, the next proposal call reads that report, and the loop runs again.

```
tasting note
    |
    v
38 TypeSafe answers
    |-- 29 score questions x 2 columns = 58   (expected rubric level + answer uncertainty)
    `--  9 noul questions  x 1 column  =  9   (probability true)
    |
    v
67 numeric columns --> CatBoost --> predicted critic score   (held-out RMSE: 1.77 points)
```

## Two question types, two column shapes

- **`intensity` -> `Score`**, for anything that comes in degrees, graded on five fixed levels (`Not present at all` ... `Dominant - the note is largely about this`). Becomes **two** columns: the mean level the answer points at, and the spread around that mean.
- **`presence` -> `Noul`**, for a yes/no fact. Becomes **one** column: the probability.

## Turning a Score's `probabilities` into mean + spread

```python
def encode(feature, probabilities, mode="mean_spread"):
    levels = np.arange(probabilities.shape[1])
    mean = probabilities @ levels
    variance = probabilities @ (levels ** 2) - mean ** 2
    return [(name, mean), (f"{name}_sd", np.sqrt(np.clip(variance, 0, None)))]
```

Ordinary weighted-mean and weighted-variance, computed straight from the same `probabilities` a `Score` answer already returns — no separate call, no extra information beyond what you already have.

## The autoresearch loop (not reproduced here)

Five rounds: propose up to 18 candidate questions from a brief plus the model's current errors, answer all of them for every row (one `Choice`/`Score`/`Noul` request per row per round — cost scales with rows, not questions), refit CatBoost, and keep an addition only if its column isn't flat, keep a revision or drop only if it actually lowers dev error. A refit costs no API calls, so trying and rejecting a change is free.

Over 2,000 wine reviews, five rounds took the question set from 18 to 38 (peaking; round 5 mostly dropped rather than added) and improved held-out RMSE by **-0.097 points** (95% CI [-0.147, -0.050]) — genuinely better than noise, if not a dramatic swing. The top-importance question by a wide margin (17.4% of total CatBoost importance) turned out to be about *tone*, not any specific tasting descriptor: *"Setting aside specific descriptors, how positive is the overall emotional tone and word choice of the note taken as a whole?"*

> The full cookbook runs the actual five-round autoresearch loop against 2,000 real wine reviews and trains a real CatBoost regressor — that's a full ML training pipeline, not a TypeSafe technique on its own, and CatBoost isn't available in this Deno/TypeScript repo. Trimmed here to the one TypeSafe-specific building block: turning a batch of Score/Noul answers into a numeric feature vector. Read the full cookbook at [docs.typesafe.ai/cookbooks/autoresearch_feature_discovery](https://docs.typesafe.ai/cookbooks/autoresearch_feature_discovery) any time.

</details>

## The core idea

`Score` and `Noul` already return exactly what a numeric feature needs — `Score`'s `probabilities` are a full distribution over an ordered scale, and its `.score` field is already the same probability-weighted mean the featurizer computes; `Noul`'s single probability is already a number in `[0, 1]`. Turning a judgment into an ML feature isn't a new capability, it's just *not throwing away* the distribution you were already given. The reason a `Score` earns a second column (the spread) and a `Noul` doesn't is the same reason from lesson 3: a `Score` carries genuine positional uncertainty across several levels, while a `Noul` is already a single number with nothing further to summarize.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { score, noul } from "npm:@typesafe-ai/sdk@^0.7.0";

// The real five-level intensity rubric and presence criteria from the cookbook.
const INTENSITY_LEVELS = [
  "Not present in this note at all",
  "Barely present - mentioned once, in passing",
  "Present at a moderate level",
  "Present strongly - the note dwells on it",
  "Dominant - the note is largely about this",
] as const;

const PRESENCE_CRITERIA = {
  true: "The note states this or clearly implies it",
  false: "The note gives no indication of this",
};

const tastingNote =
  "A brooding, structured red with firm, grippy tannins that dominate the palate through a " +
  "long finish. Dark fruit sits underneath but never takes the lead. No flaws to speak of, " +
  "and this comes from a single, well-regarded vineyard block.";

// Two real question names from the cookbook's own round 1 (score-style), plus two more
// (noul-style) - a tiny slice of the 38 it eventually kept.
const response = await client.systemOne({
  state: tastingNote,
  questions: {
    tanninStructure: score(
      "How much does tannin structure feature in this note?",
      INTENSITY_LEVELS,
    ),
    fruitIntensity: score(
      "How much does fruit character feature in this note?",
      INTENSITY_LEVELS,
    ),
    flawMentioned: noul("Does the note mention a flaw or defect in the wine?", PRESENCE_CRITERIA),
    prestigeSignal: noul(
      "Does the note signal a single-vineyard or prestige bottling?",
      PRESENCE_CRITERIA,
    ),
  },
});

const { tanninStructure, fruitIntensity, flawMentioned, prestigeSignal } = response.answers;

// A Score becomes two numeric columns: the mean level, and the spread (sd) around it -
// the same weighted-mean / weighted-variance the cookbook's encode() computes,
// straight from `probabilities`.
function meanAndSpread(probabilities: Record<string, number>): { mean: number; sd: number } {
  const levels = Object.keys(probabilities).map(Number);
  const mean = levels.reduce((sum, level) => sum + level * probabilities[String(level)], 0);
  const variance =
    levels.reduce((sum, level) => sum + level * level * probabilities[String(level)], 0) -
    mean * mean;
  return { mean, sd: Math.sqrt(Math.max(variance, 0)) };
}

const tannin = meanAndSpread(tanninStructure.probabilities);
const fruit = meanAndSpread(fruitIntensity.probabilities);

// The numeric feature vector: this row is now ready for an ML model that needs numbers,
// not text - a Noul contributes 1 column, a Score contributes 2.
const featureVector = {
  tannin_structure: tannin.mean,
  tannin_structure_sd: tannin.sd,
  fruit_intensity: fruit.mean,
  fruit_intensity_sd: fruit.sd,
  flaw_or_defect_mentioned: flawMentioned.noul,
  single_vineyard_or_prestige_signal: prestigeSignal.noul,
};

console.log("feature vector:", featureVector);

## What just happened

One tasting note turned into six numeric columns from four questions — exactly the shape the real pipeline scales up to 2,000 rows and 38 questions. `tannin_structure` should land high (the note explicitly says tannins "dominate the palate") with low spread (the note isn't ambiguous about it); `fruit_intensity` should land lower, since the note explicitly says fruit "never takes the lead." Both `Noul`s should read close to `1.0` — the note directly states there are no flaws and directly mentions a "single... vineyard block."

**Try it yourself:** rewrite the note to be genuinely ambiguous about tannin level (e.g. mention grip in one sentence and softness in another) and predict whether `tannin_structure_sd` rises noticeably — a real jump in spread is exactly the signal CatBoost would use to distrust that column on an ambiguous row.

## Related

- `lessons/core-concepts/0003-score.ipynb` — why `Score` carries a spread worth computing and `Noul` doesn't
- `cookbooks/extraction/sde-cascade.ipynb` — another cookbook that reads a Score/Noul battery's raw numbers programmatically rather than just the top answer

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.